In [1]:
!pip install -q openai-whisper jiwer

In [2]:
import os, re, json, requests, warnings
import pandas as pd
import torch
import torchaudio
import whisper
from jiwer import wer
from pathlib import Path
from urllib.parse import urlparse

warnings.filterwarnings("ignore")
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [3]:
CSV_PATH = "/kaggle/input/YOUR-DATASET-NAME/FT_Data_-_data.csv"  # <-- change this

df = pd.read_csv(CSV_PATH)
print(f"Total recordings: {len(df)}")

BASE_NEW = "https://storage.googleapis.com/upload_goai"

def fix_url(original_url):
    parts = urlparse(original_url).path.strip("/").split("/")
    user_id  = parts[-2]
    filename = parts[-1]
    return f"{BASE_NEW}/{user_id}/{filename}"

df["trans_url_fixed"] = df["transcription_url_gcp"].apply(fix_url)
df["audio_url_fixed"] = df["rec_url_gcp"].apply(fix_url)

test = requests.get(df["trans_url_fixed"].iloc[0], timeout=10)
print("Status:", test.status_code)
print("Sample JSON:", test.json())

Total recordings: 104
Status: 200
Sample JSON: [{'start': 0.11, 'end': 14.42, 'speaker_id': 245746, 'text': 'अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब'}, {'start': 14.42, 'end': 29.03, 'speaker_id': 245746, 'text': 'अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नहीं हो सकती थी तो हम वहां गया थे कुड़रमा घाटी तरफ पर दिवोग काफी जंगली एरिया है वह जो खांड जनजाति पाए जाती ना वहां पाए जाती है तो'}, {'start': 29.03, 'end': 41.84, 'speaker_id': 245746, 'text': 'जंगल का सफर होता है जब हम रहने के लिए गए थे नातो चाहते के साथ जैसे हम वहाँ पहले एंटर किये थे तो पहले तो गिर गया थे लगड़ा के उपर से नीचे'}, {'start': 42.47, 'end': 50.57, 'speaker_id': 245746, 'text': 'पहली बारी था क्योंकि चलना नहीं आता न वहाँ का जो लैंड एरिया होता है इधर उधर काफी इधर जाओ तो उधर लुढ़क जाओगे'}, {'start': 52.7, 'end': 66.83, 'speaker_id': 245746, '

In [4]:
def fetch_transcription(url):
    try:
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            return None
        data = r.json()
        for key in ["transcription", "text", "transcript", "content"]:
            if key in data:
                return data[key].strip()
        if isinstance(data, list):
            return " ".join([seg.get("text", "") for seg in data]).strip()
        return str(data)
    except Exception as e:
        print(f"Error: {e}")
        return None

def trim_reference(text, max_words=60):
    # ~60 words ≈ 30 seconds of speech — keeps WER comparison fair
    if not text:
        return text
    return " ".join(text.split()[:max_words])

print("Fetching reference transcriptions...")
df["reference"] = df["trans_url_fixed"].apply(fetch_transcription)
df["reference"] = df["reference"].apply(trim_reference)
df = df.dropna(subset=["reference"]).reset_index(drop=True)
print(f"Got {len(df)} transcriptions")

Fetching reference transcriptions...
Got 104 transcriptions


In [5]:
SAMPLE_SIZE = 30
AUDIO_DIR = Path("/kaggle/working/audio_cache")
AUDIO_DIR.mkdir(exist_ok=True)

sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)

def download_audio(url, save_path):
    if save_path.exists():
        return True
    try:
        r = requests.get(url, timeout=60, stream=True)
        if r.status_code == 200:
            with open(save_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            return True
    except Exception as e:
        print(f"Download error: {e}")
    return False

def load_first_n_seconds(audio_path, seconds=30, target_sr=16000):
    waveform, sr = torchaudio.load(str(audio_path))
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(sr, target_sr)
        waveform = resampler(waveform)
    max_samples = seconds * target_sr
    waveform = waveform[:, :max_samples]
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    return waveform.squeeze().numpy()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Whisper-small on {device}...")
model = whisper.load_model("small", device=device)

raw_asr_outputs = []
for idx, row in sample_df.iterrows():
    audio_path = AUDIO_DIR / f"{row['recording_id']}.wav"
    ok = download_audio(row["audio_url_fixed"], audio_path)
    if not ok:
        raw_asr_outputs.append(None)
        continue
    try:
        audio_array = load_first_n_seconds(audio_path, seconds=30)
        result = model.transcribe(
            audio_array,
            language="hi",
            task="transcribe",
            fp16=(device == "cuda"),
            beam_size=5,
            best_of=5,
            temperature=0.0
        )
        raw_asr_outputs.append(result["text"].strip())
        print(f"[{idx+1}/{len(sample_df)}] {result['text'][:80]}")
    except Exception as e:
        print(f"Error: {e}")
        raw_asr_outputs.append(None)

sample_df["raw_asr"] = raw_asr_outputs
sample_df = sample_df.dropna(subset=["raw_asr"]).reset_index(drop=True)
print(f"\nDone. Got ASR for {len(sample_df)} recordings.")

Loading Whisper-small on cuda...
[1/30]  तो सर अक्छली आप चाहता आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप आप 
[2/30]  किर्किट के बारे में किर्किट मैं 11 अखेलाडी होते होते हैं और लिए बांड़़ी होते है
[3/30]  या से क्रिकेट क्रिकेट के बारे में बतागे सूँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँ
[4/30]  अंजी दिलकु अंजी अंजी दिलकु
[5/30]  अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे
[6/30]  तो स्नेक्स वगरा तो जोरूर होने जीए साथ में तो स्नेक्स में भी अपनी अपनी जोई साथ ह
[7/30]  सर जब ये होता है, जब गिरते है, जब जादा चोट लगती है, तो वहां से फिरा जाती है, सै
[8/30]  आप आप इस्टार्टिं करीए फिर मुँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँ
[9/30]  जी तो कोई है आसी जगे जगे जगाए मुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुुु
[10/30]  अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने
[11/30]  तो सरएसर भी वो ता कभी की आप देरा तक पडे हो तो सर्कोलेस टाई में अपने जो प्रुफेशर
[12/30]  तो थीके मैं अपनी जो गड़ना है मैं बताता हो

In [6]:
UNITS = {
    "शून्य":0,"एक":1,"दो":2,"तीन":3,"चार":4,"पाँच":5,"पांच":5,
    "छह":6,"छः":6,"सात":7,"आठ":8,"नौ":9,"दस":10,"ग्यारह":11,
    "बारह":12,"तेरह":13,"चौदह":14,"पंद्रह":15,"सोलह":16,"सत्रह":17,
    "अठारह":18,"उन्नीस":19,"बीस":20,"इक्कीस":21,"बाईस":22,"तेईस":23,
    "चौबीस":24,"पच्चीस":25,"छब्बीस":26,"सत्ताईस":27,"अट्ठाईस":28,
    "उनतीस":29,"तीस":30,"इकतीस":31,"बत्तीस":32,"तेंतीस":33,"चौंतीस":34,
    "पैंतीस":35,"छत्तीस":36,"सैंतीस":37,"अड़तीस":38,"उनतालीस":39,
    "चालीस":40,"इकतालीस":41,"बयालीस":42,"तैंतालीस":43,"चौवालीस":44,
    "पैंतालीस":45,"छयालीस":46,"सैंतालीस":47,"अड़तालीस":48,"उनचास":49,
    "पचास":50,"इक्यावन":51,"बावन":52,"तिरपन":53,"चौवन":54,"पचपन":55,
    "छप्पन":56,"सत्तावन":57,"अट्ठावन":58,"उनसठ":59,"साठ":60,
    "इकसठ":61,"बासठ":62,"तिरसठ":63,"चौसठ":64,"पैंसठ":65,
    "छियासठ":66,"सड़सठ":67,"अड़सठ":68,"उनहत्तर":69,"सत्तर":70,
    "इकहत्तर":71,"बहत्तर":72,"तिहत्तर":73,"चौहत्तर":74,"पचहत्तर":75,
    "छिहत्तर":76,"सतहत्तर":77,"अठहत्तर":78,"उनासी":79,"अस्सी":80,
    "इक्यासी":81,"बयासी":82,"तिरासी":83,"चौरासी":84,"पचासी":85,
    "छियासी":86,"सत्तासी":87,"अठासी":88,"नवासी":89,"नब्बे":90,
    "इक्यानवे":91,"बानवे":92,"तिरानवे":93,"चौरानवे":94,"पचानवे":95,
    "छियानवे":96,"सत्तानवे":97,"अट्ठानवे":98,"निन्यानवे":99,
}

MULTIPLIERS = {
    "सौ":100,
    "हज़ार":1000,"हजार":1000,
    "लाख":100000,
    "करोड़":10000000,"करोड":10000000,
}

IDIOMS = {
    "दो-चार","दो चार","तीन-चार","तीन चार",
    "दो-तीन","दो तीन","एक-दो","एक दो",
    "चार-पाँच","चार पांच","पाँच-सात","पांच सात"
}

def is_idiom(tokens, i):
    if i + 1 < len(tokens):
        if tokens[i] + "-" + tokens[i+1] in IDIOMS:
            return True
        if tokens[i] + " " + tokens[i+1] in IDIOMS:
            return True
    if "-" in tokens[i]:
        parts = tokens[i].split("-")
        if all(p in UNITS for p in parts):
            return True
    return False

def parse_number(tokens, start):
    i, value, found = start, 0, False
    while i < len(tokens):
        tok = tokens[i].strip("।,")
        if tok in UNITS:
            uval = UNITS[tok]
            if i+1 < len(tokens) and tokens[i+1].strip("।,") in MULTIPLIERS:
                value += uval * MULTIPLIERS[tokens[i+1].strip("।,")]
                i += 2
            else:
                value += uval
                i += 1
            found = True
        elif tok in MULTIPLIERS:
            value = (value if value else 1) * MULTIPLIERS[tok]
            i += 1
            found = True
        else:
            break
    return (value, i) if found else (None, start)

def normalize_numbers(text):
    tokens = text.split()
    out, changes = [], []
    i = 0
    while i < len(tokens):
        tok = tokens[i].strip("।,")
        if tok in UNITS or tok in MULTIPLIERS:
            if is_idiom(tokens, i):
                out.append(tokens[i])
                changes.append((tokens[i], tokens[i], "IDIOM_KEPT"))
                i += 1
            else:
                val, end = parse_number(tokens, i)
                if val is not None:
                    original = " ".join(tokens[i:end])
                    out.append(str(val))
                    changes.append((original, str(val), "CONVERTED"))
                    i = end
                else:
                    out.append(tokens[i])
                    i += 1
        else:
            out.append(tokens[i])
            i += 1
    return " ".join(out), changes

sample_df["normalized"] = sample_df["raw_asr"].apply(lambda t: normalize_numbers(t)[0])
sample_df["norm_changes"] = sample_df["raw_asr"].apply(lambda t: normalize_numbers(t)[1])

print("="*60)
print("PART A — NUMBER NORMALIZATION")
print("="*60)

shown = 0
for _, row in sample_df.iterrows():
    for orig, conv, tag in row["norm_changes"]:
        if tag == "CONVERTED" and shown < 5:
            print(f"\nBEFORE : {row['raw_asr'][:100]}")
            print(f"AFTER  : {row['normalized'][:100]}")
            print(f"CHANGE : '{orig}' → '{conv}'")
            shown += 1

print("\n--- DEMO EXAMPLES ---")
demos = [
    "मैंने दस किताबें पढ़ीं",
    "तीन सौ चौवन रुपये दिए",
    "एक हज़ार लोग आए थे",
    "पच्चीस साल पहले की बात है",
    "दो-चार बातें करनी हैं",
    "एक-दो दिन में आ जाऊंगा",
    "उसने दो लाख कमाए",
    "पाँच सौ रुपये का नोट",
    "तीस हज़ार की तनख्वाह",
]
for s in demos:
    norm, changes = normalize_numbers(s)
    print(f"\nIN  : {s}")
    print(f"OUT : {norm}")
    for orig, conv, tag in changes:
        print(f"      [{tag}] '{orig}' → '{conv}'")

PART A — NUMBER NORMALIZATION

BEFORE : मेरा सपनो का इवेंट है, जिस में मुझे जाना है, मुझे अप एक बारी मुझे क्रिकिट देख में जाना है, लाइप क्रि
AFTER  : मेरा सपनो का इवेंट है, जिस में मुझे जाना है, मुझे अप 1 बारी मुझे क्रिकिट देख में जाना है, लाइप क्रिक
CHANGE : 'एक' → '1'

BEFORE : तो मैंने पर्वार में उसके बारे में बताया है कि मैं ये नोकरी चोड़ रहा हूं कि उसरे में काफी समय से एक व
AFTER  : तो मैंने पर्वार में उसके बारे में बताया है कि मैं ये नोकरी चोड़ रहा हूं कि उसरे में काफी समय से 1 वर
CHANGE : 'एक' → '1'

--- DEMO EXAMPLES ---

IN  : मैंने दस किताबें पढ़ीं
OUT : मैंने 10 किताबें पढ़ीं
      [CONVERTED] 'दस' → '10'

IN  : तीन सौ चौवन रुपये दिए
OUT : 354 रुपये दिए
      [CONVERTED] 'तीन सौ चौवन' → '354'

IN  : एक हज़ार लोग आए थे
OUT : 1000 लोग आए थे
      [CONVERTED] 'एक हज़ार' → '1000'

IN  : पच्चीस साल पहले की बात है
OUT : 25 साल पहले की बात है
      [CONVERTED] 'पच्चीस' → '25'

IN  : दो-चार बातें करनी हैं
OUT : दो-चार बातें करनी हैं

IN  : एक-दो दिन में आ जाऊंगा
OUT : एक-दो दिन में 

In [7]:
DEVANAGARI_LOANWORDS = {
    "इंटरव्यू","जॉब","ऑफिस","मैनेजर","टीम","प्रोजेक्ट","डेडलाइन",
    "मीटिंग","प्रेजेंटेशन","रिपोर्ट","ईमेल","लैपटॉप","कंप्यूटर",
    "फोन","मोबाइल","इंटरनेट","स्कूल","कॉलेज","क्लास","टेस्ट",
    "एग्जाम","सिलेबस","नोट्स","असाइनमेंट","प्रोफेसर","ओके",
    "थैंक्यू","सॉरी","प्लीज़","हेलो","बाय","हाय","फेसबुक",
    "व्हाट्सएप","यूट्यूब","गूगल","ट्विटर","वीडियो","ऑनलाइन",
    "ऑफलाइन","डाउनलोड","अपलोड","बैंक","लोन","पेमेंट","सैलरी",
    "टाइम","सिस्टम","प्रॉब्लम","सॉल्यूशन","रिज़ल्ट","सक्सेस",
    "पॉज़िटिव","नेगेटिव","पर्सनल","प्रोफेशनल","नॉर्मल","स्पेशल",
    "ट्रेंड","वायरल","ऐप","चैट","पोस्ट","लाइक","शेयर","फॉलो",
}

KNOWN_HINGLISH_ROMAN = {
    "ok","okay","hi","hello","bye","yes","no","sir","mam","madam",
    "please","sorry","thanks","thank","you","ji","bhi",
    "interview","job","office","manager","team","project","deadline",
    "meeting","presentation","report","email","laptop","computer",
    "phone","mobile","internet","school","college","class","test",
    "exam","online","offline","download","upload","bank","loan",
    "payment","salary","time","system","problem","solution","result",
    "success","positive","negative","personal","professional",
    "normal","special","youtube","google","facebook","whatsapp",
    "twitter","video","free","option","choice","pass","fail",
    "cricket","match","score","over","run","wicket","ball","bat",
    "app","data","file","pdf","link","share","post","like","follow",
    "trend","viral","chat","comment","profile","status","update",
    "train","bus","bike","car","drive","ticket","station","platform",
    "doctor","hospital","medicine","tablet","dose","test","report",
    "police","court","case","law","rule","fine","tax","form",
}

def has_devanagari(word):
    return bool(re.search(r'[\u0900-\u097F]', word))

def is_pure_roman(word):
    clean = re.sub(r'[^A-Za-z\u0900-\u097F]', '', word)
    if not clean:
        return False
    return bool(re.search(r'[A-Za-z]', clean)) and not has_devanagari(clean)

def is_known_english(word):
    return word.lower().strip(".,!?;:") in KNOWN_HINGLISH_ROMAN

def is_deva_english(word):
    return re.sub(r'[।,!?\s]', '', word) in DEVANAGARI_LOANWORDS

def tag_english(text):
    tokens = text.split()
    out, detected = [], []
    for tok in tokens:
        clean = tok.strip("।,!()?;:")
        if is_pure_roman(clean) and is_known_english(clean):
            out.append(f"[EN]{tok}[/EN]")
            detected.append((tok, "ROMAN"))
        elif is_deva_english(clean):
            out.append(f"[EN:LOAN]{tok}[/EN:LOAN]")
            detected.append((tok, "LOANWORD"))
        else:
            out.append(tok)
    return " ".join(out), detected

sample_df["tagged"] = sample_df["normalized"].apply(lambda t: tag_english(t)[0])
sample_df["english_detected"] = sample_df["normalized"].apply(lambda t: tag_english(t)[1])

print("="*60)
print("PART B — ENGLISH WORD DETECTION")
print("="*60)

shown = 0
for _, row in sample_df.iterrows():
    if row["english_detected"] and shown < 5:
        print(f"\nINPUT  : {row['normalized'][:120]}")
        print(f"TAGGED : {row['tagged'][:140]}")
        print(f"FOUND  : {row['english_detected']}")
        shown += 1

if shown == 0:
    print("\nNo English words in sample. See demos below:")

print("\n--- DEMO EXAMPLES ---")
demo_tag = [
    "मेरा इंटरव्यू बहुत अच्छा गया और मुझे जॉब मिल गई",
    "मेरा interview अच्छा गया",
    "यह problem solve नहीं हो रहा",
    "मैं online class ले रहा हूँ",
    "उसने कंप्यूटर पर काम किया",
    "ok sir मैं समझ गया",
    "उसका cricket match था",
    "मेरा phone का battery low है",
    "उसने youtube पर video देखी",
]
for s in demo_tag:
    tagged, detected = tag_english(s)
    print(f"\nIN  : {s}")
    print(f"OUT : {tagged}")
    print(f"EN  : {[w for w,_ in detected]}")

PART B — ENGLISH WORD DETECTION

No English words in sample. See demos below:

--- DEMO EXAMPLES ---

IN  : मेरा इंटरव्यू बहुत अच्छा गया और मुझे जॉब मिल गई
OUT : मेरा [EN:LOAN]इंटरव्यू[/EN:LOAN] बहुत अच्छा गया और मुझे [EN:LOAN]जॉब[/EN:LOAN] मिल गई
EN  : ['इंटरव्यू', 'जॉब']

IN  : मेरा interview अच्छा गया
OUT : मेरा [EN]interview[/EN] अच्छा गया
EN  : ['interview']

IN  : यह problem solve नहीं हो रहा
OUT : यह [EN]problem[/EN] solve नहीं हो रहा
EN  : ['problem']

IN  : मैं online class ले रहा हूँ
OUT : मैं [EN]online[/EN] [EN]class[/EN] ले रहा हूँ
EN  : ['online', 'class']

IN  : उसने कंप्यूटर पर काम किया
OUT : उसने [EN:LOAN]कंप्यूटर[/EN:LOAN] पर काम किया
EN  : ['कंप्यूटर']

IN  : ok sir मैं समझ गया
OUT : [EN]ok[/EN] [EN]sir[/EN] मैं समझ गया
EN  : ['ok', 'sir']

IN  : उसका cricket match था
OUT : उसका [EN]cricket[/EN] [EN]match[/EN] था
EN  : ['cricket', 'match']

IN  : मेरा phone का battery low है
OUT : मेरा [EN]phone[/EN] का battery low है
EN  : ['phone']

IN  : उसने youtube पर video देखी

In [8]:
from jiwer import wer as compute_wer

def strip_tags(text):
    return re.sub(r'\[/?EN(?::LOAN)?\]', '', text).strip()

sample_df["tagged_clean"] = sample_df["tagged"].apply(strip_tags)

def safe_wer(refs, hyps):
    pairs = [(r,h) for r,h in zip(refs,hyps) if r and h and r.strip() and h.strip()]
    if not pairs:
        return float("nan")
    r, h = zip(*pairs)
    return compute_wer(list(r), list(h))

wer_raw  = safe_wer(sample_df["reference"], sample_df["raw_asr"])
wer_norm = safe_wer(sample_df["reference"], sample_df["normalized"])
wer_pipe = safe_wer(sample_df["reference"], sample_df["tagged_clean"])

print("\n" + "="*52)
print(f"{'Stage':<35} {'WER':>8}  {'% Change':>10}")
print("-"*52)
print(f"{'Raw Whisper-small (baseline)':<35} {wer_raw:>8.4f}  {'—':>10}")
print(f"{'After Number Normalization':<35} {wer_norm:>8.4f}  {((wer_norm-wer_raw)/wer_raw*100):>+9.2f}%")
print(f"{'After Full Pipeline':<35} {wer_pipe:>8.4f}  {((wer_pipe-wer_raw)/wer_raw*100):>+9.2f}%")
print("="*52)

print("""
Note:
- Only first 30 seconds of each recording used (for speed)
- Reference trimmed to ~60 words to match ASR scope
- WER increase after normalization is expected: references use
  word-form numbers (दस) while pipeline outputs digits (10).
  A fair eval requires normalizing references too, or using
  lattice-based WER (see Q4).
- EN tagging does not change WER (tags stripped before scoring).
""")


Stage                                    WER    % Change
----------------------------------------------------
Raw Whisper-small (baseline)          0.9628           —
After Number Normalization            0.9633      +0.06%
After Full Pipeline                   0.9633      +0.06%

Note:
- Only first 30 seconds of each recording used (for speed)
- Reference trimmed to ~60 words to match ASR scope
- WER increase after normalization is expected: references use
  word-form numbers (दस) while pipeline outputs digits (10).
  A fair eval requires normalizing references too, or using
  lattice-based WER (see Q4).
- EN tagging does not change WER (tags stripped before scoring).



In [9]:
sample_df[[
    "recording_id","reference","raw_asr",
    "normalized","tagged","tagged_clean","english_detected"
]].to_csv("/kaggle/working/q2_output.csv", index=False)

print("Saved → /kaggle/working/q2_output.csv")
print("Done ✓")

Saved → /kaggle/working/q2_output.csv
Done ✓


In [10]:
import pandas as pd
df = pd.read_csv('q2_output.csv')
print(df.head())

   recording_id                                          reference  \
0        615351  तो सर एक्चुली अ आप चाहें तो आप पहले डिस्क्राइब...   
1        302503  क्या पैकेट में क्या रहता है। हा हा ह क्रिकेट क...   
2        302506  यस क्रिकेट,क्रिकेट क्रिकेट के बारे मै बताइये न...   
3        630926  हांजी बिल्कुल उम हा हां हां बिल्कुल हां हम्म ह...   
4        350347  फिर ओके अच्छा जी जी। ओके हम्मम हम्मम सही बात ज...   

                                             raw_asr  \
0  तो सर अक्छली आप चाहता आप आप आप आप आप आप आप आप ...   
1  किर्किट के बारे में किर्किट मैं 11 अखेलाडी होत...   
2  या से क्रिकेट क्रिकेट के बारे में बतागे सूँँँँ...   
3                         अंजी दिलकु अंजी अंजी दिलकु   
4  अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे अखे अख...   

                                          normalized  \
0  तो सर अक्छली आप चाहता आप आप आप आप आप आप आप आप ...   
1  किर्किट के बारे में किर्किट मैं 11 अखेलाडी होत...   
2  या से क्रिकेट क्रिकेट के बारे में बतागे सूँँँँ...   
3                 